<a href="https://colab.research.google.com/github/awinarko-hue/Project_hue/blob/main/OnWork/scientific_literature_retrieval_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG / Scientific Literature Retrieval Experiment v2.0
**BM25 + Dense FAISS + Hybrid RRF + Cross-Encoder Reranking**

Evaluasi: Precision@K, Recall@K, MRR, nDCG@K

Notebook ini dirancang untuk hasil ekspor CSV dari **Dimensions** dan eksperimen tesis/publikasi.


## 0. Instalasi Dependensi (Colab)
Jalankan sel ini terlebih dahulu di Google Colab.

In [1]:
!pip install -q pandas numpy scikit-learn sentence-transformers faiss-cpu rank_bm25 tqdm matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.3 MB/s eta 0:00:00


## 0.1 Mount Google Drive
Diperlukan jika file CSV (Dimensions export, qrels, dsb.) disimpan di Google Drive Anda.
Sesuaikan `csv_path` dan `qrels_path` pada bagian **Config** di bawah setelah Drive ter-mount.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Reproducibility + Config
Import pustaka dan definisi konfigurasi eksperimen (`Config` dataclass).

In [3]:
from __future__ import annotations

import os
import re
import json
import math
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Iterable, Optional

import numpy as np
import pandas as pd
import faiss
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

@dataclass
class Config:
    # Change to your Dimensions CSV path
    csv_path: str = "/content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/Penulisan/Dimensions-Publication-Graph_RAG.csv"

    # Optional publication-grade relevance judgments.
    # Required columns:
    # query_id, doc_id, relevance
    # relevance: 0=not relevant, 1=partly, 2=relevant, 3=highly relevant
    qrels_path: str = "/content/drive/MyDrive/Dataset/Paper/qrels_lte.csv"

    output_dir: str = "/content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/Penulisan//retrieval_v2_results"

    # Dense bi-encoder.
    # E5 expects "query:" and "passage:" prefixes.
    dense_model: str = "intfloat/e5-base-v2"

    # Cross-encoder reranker
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L6-v2"

    # Retrieval depths
    bm25_top_k: int = 100
    dense_top_k: int = 100
    hybrid_top_k: int = 50
    rerank_top_k: int = 10

    # RRF constant; 60 is a common robust default.
    rrf_k: int = 60

    # Batch sizes
    embedding_batch_size: int = 32
    rerank_batch_size: int = 32

    # Metrics
    eval_ks: Tuple[int, ...] = (5, 10, 20)

    # For publication evaluation, relevance >= threshold is binary relevant
    binary_relevance_threshold: int = 2

    # CSV parsing inherited from the original notebook
    skiprows: int = 1
    encoding: str = "latin-1"

CFG = Config()
os.makedirs(CFG.output_dir, exist_ok=True)

print("CONFIG:")
print(json.dumps(asdict(CFG), indent=2, default=list))

CONFIG:
{
  "csv_path": "/content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/Penulisan/Dimensions-Publication-Graph_RAG.csv",
  "qrels_path": "/content/drive/MyDrive/Dataset/Paper/qrels_lte.csv",
  "output_dir": "/content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/Penulisan//retrieval_v2_results",
  "dense_model": "intfloat/e5-base-v2",
  "reranker_model": "cross-encoder/ms-marco-MiniLM-L6-v2",
  "bm25_top_k": 100,
  "dense_top_k": 100,
  "hybrid_top_k": 50,
  "rerank_top_k": 10,
  "rrf_k": 60,
  "embedding_batch_size": 32,
  "rerank_batch_size": 32,
  "eval_ks": [
    5,
    10,
    20
  ],
  "binary_relevance_threshold": 2,
  "skiprows": 1,
  "encoding": "latin-1"
}


## 2. Data Loading + Quality Checks
Memuat CSV hasil ekspor Dimensions dan membersihkan kolom bibliografi yang dibutuhkan untuk retrieval.

In [5]:
def load_dimensions_csv(path: str, skiprows: int = 1, encoding: str = "latin-1") -> pd.DataFrame:
    """
    Load a Dimensions CSV and keep bibliographic fields needed for retrieval.
    """
    df = pd.read_csv(
        path,
        skiprows=skiprows,
        encoding=encoding,
        on_bad_lines="warn",
        low_memory=False,
    )

    desired = [
        "Publication ID",
        "DOI",
        "Title",
        "Abstract",
        "Authors",
        "PubYear",
        "Source title/Anthology title",
        "Dimensions URL",
    ]

    missing_required = [c for c in ["Publication ID", "Title", "Abstract"] if c not in df.columns]
    if missing_required:
        raise ValueError(f"Required columns missing: {missing_required}")

    # Add optional fields if absent
    for col in desired:
        if col not in df.columns:
            df[col] = ""

    df = df[desired].copy()

    # Basic cleaning
    df["Title"] = df["Title"].fillna("").astype(str).str.strip()
    df["Abstract"] = df["Abstract"].fillna("").astype(str).str.strip()
    df["DOI"] = df["DOI"].fillna("").astype(str).str.strip()
    df["Authors"] = df["Authors"].fillna("").astype(str).str.strip()
    df["PubYear"] = df["PubYear"].fillna("").astype(str).str.strip()
    df["Source title/Anthology title"] = (
        df["Source title/Anthology title"].fillna("").astype(str).str.strip()
    )
    df["Dimensions URL"] = df["Dimensions URL"].fillna("").astype(str).str.strip()
    df["Publication ID"] = df["Publication ID"].fillna("").astype(str).str.strip()

    # Drop records without titles
    df = df[df["Title"] != ""].copy()

    # Build stable document ID:
    # DOI > Publication ID > normalized title
    norm_title = (
        df["Title"]
        .str.lower()
        .str.replace(r"\W+", "_", regex=True)
        .str.strip("_")
        .str.slice(0, 100)
    )
    df["doc_id"] = np.where(
        df["DOI"] != "",
        "doi:" + df["DOI"].str.lower(),
        np.where(
            df["Publication ID"] != "",
            "pub:" + df["Publication ID"],
            "title:" + norm_title,
        ),
    )

    # Deduplicate documents
    before = len(df)
    df = df.drop_duplicates(subset=["doc_id"], keep="first").reset_index(drop=True)
    after = len(df)

    # One paper = one retrieval document.
    # This avoids chunk duplication for title+abstract retrieval.
    df["retrieval_text"] = (
        "Title: " + df["Title"] + "\nAbstract: " + df["Abstract"]
    )

    print(f"Rows loaded       : {before}")
    print(f"Unique documents  : {after}")
    print(f"Duplicates removed: {before - after}")
    print(f"Missing abstracts : {(df['Abstract'] == '').sum()}")
    print(f"Missing DOI       : {(df['DOI'] == '').sum()}")

    return df

## 3. Text Normalization untuk BM25

In [6]:
TOKEN_RE = re.compile(r"[a-zA-Z0-9][a-zA-Z0-9_\-/+.]*")

def bm25_tokenize(text: str) -> List[str]:
    """
    Conservative tokenizer for scientific/telecom terminology.
    Keeps tokens such as 4g, 5g, rsrp, sinr, machine-learning-like strings.
    """
    text = text.lower()
    return TOKEN_RE.findall(text)

## 4. BM25 Retriever

In [7]:
class BM25Retriever:
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
        self.tokenized_corpus = [
            bm25_tokenize(x) for x in tqdm(self.df["retrieval_text"], desc="BM25 tokenize")
        ]
        self.index = BM25Okapi(self.tokenized_corpus)

    def search(self, query: str, top_k: int = 100) -> pd.DataFrame:
        q_tokens = bm25_tokenize(query)
        scores = np.asarray(self.index.get_scores(q_tokens), dtype=float)

        top_k = min(top_k, len(scores))
        # argpartition + sort is more efficient than sorting the whole corpus
        idx = np.argpartition(-scores, top_k - 1)[:top_k]
        idx = idx[np.argsort(-scores[idx])]

        out = self.df.loc[idx, ["doc_id", "Title"]].copy()
        out["score"] = scores[idx]
        out["rank"] = np.arange(1, len(out) + 1)
        out["method"] = "BM25"
        return out.reset_index(drop=True)

## 5. Dense Retriever + FAISS
Menggunakan bi-encoder (default: `intfloat/e5-base-v2`) dan indeks FAISS (`IndexFlatIP`).

In [8]:
class DenseFAISSRetriever:
    def __init__(
        self,
        df: pd.DataFrame,
        model_name: str,
        batch_size: int = 32,
    ):
        self.df = df.reset_index(drop=True)
        self.model_name = model_name
        self.batch_size = batch_size

        print(f"Loading dense model: {model_name}")
        self.model = SentenceTransformer(model_name)

        passages = ["passage: " + x for x in self.df["retrieval_text"].tolist()]
        print("Encoding corpus...")
        embeddings = self.model.encode(
            passages,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")

        self.embeddings = embeddings

        # Cosine similarity == inner product when vectors are L2-normalized
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)

        print(f"FAISS vectors: {self.index.ntotal}, dimension: {dim}")

    def search(self, query: str, top_k: int = 100) -> pd.DataFrame:
        q = self.model.encode(
            ["query: " + query],
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")

        top_k = min(top_k, len(self.df))
        scores, indices = self.index.search(q, top_k)

        idx = indices[0]
        sim = scores[0]

        valid = idx >= 0
        idx, sim = idx[valid], sim[valid]

        out = self.df.loc[idx, ["doc_id", "Title"]].copy()
        out["score"] = sim
        out["rank"] = np.arange(1, len(out) + 1)
        out["method"] = "Dense"
        return out.reset_index(drop=True)

## 6. Hybrid Retrieval dengan Reciprocal Rank Fusion (RRF)

In [9]:
def reciprocal_rank_fusion(
    result_frames: List[pd.DataFrame],
    rrf_k: int = 60,
    top_k: int = 50,
) -> pd.DataFrame:
    """
    RRF combines rankings without requiring BM25 and dense scores
    to be numerically comparable.
    """
    scores: Dict[str, float] = {}
    titles: Dict[str, str] = {}

    for frame in result_frames:
        for _, row in frame.iterrows():
            doc_id = row["doc_id"]
            rank = int(row["rank"])
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank)
            titles[doc_id] = row["Title"]

    rows = [
        {"doc_id": doc_id, "Title": titles[doc_id], "score": score}
        for doc_id, score in scores.items()
    ]

    out = (
        pd.DataFrame(rows)
        .sort_values("score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )
    out["rank"] = np.arange(1, len(out) + 1)
    out["method"] = "Hybrid_RRF"
    return out

## 7. Cross-Encoder Reranker

In [10]:
class CrossEncoderReranker:
    def __init__(self, model_name: str, batch_size: int = 32):
        print(f"Loading reranker: {model_name}")
        self.model = CrossEncoder(model_name)
        self.batch_size = batch_size

    def rerank(
        self,
        query: str,
        candidates: pd.DataFrame,
        corpus_df: pd.DataFrame,
        top_k: int = 10,
    ) -> pd.DataFrame:
        lookup = corpus_df.set_index("doc_id")["retrieval_text"].to_dict()

        pairs = [
            (query, lookup[doc_id])
            for doc_id in candidates["doc_id"].tolist()
        ]

        scores = self.model.predict(
            pairs,
            batch_size=self.batch_size,
            show_progress_bar=False,
        )
        scores = np.asarray(scores, dtype=float).reshape(-1)

        out = candidates[["doc_id", "Title"]].copy()
        out["score"] = scores
        out = (
            out.sort_values("score", ascending=False)
            .head(top_k)
            .reset_index(drop=True)
        )
        out["rank"] = np.arange(1, len(out) + 1)
        out["method"] = "Hybrid_RRF+Reranker"
        return out

## 8. Qrels / Ground-Truth Format

In [11]:
def load_qrels(path: str) -> pd.DataFrame:
    """
    Expected CSV:
        query_id,doc_id,relevance
        Q1,doi:10.xxxx/abcd,3
        Q1,pub:pub.123456,2
        Q1,doi:10.xxxx/irrelevant,0

    Recommended:
      3 = highly relevant
      2 = relevant
      1 = partially relevant
      0 = not relevant
    """
    qrels = pd.read_csv(path)
    required = {"query_id", "doc_id", "relevance"}
    missing = required - set(qrels.columns)
    if missing:
        raise ValueError(f"qrels missing columns: {sorted(missing)}")

    qrels["query_id"] = qrels["query_id"].astype(str)
    qrels["doc_id"] = qrels["doc_id"].astype(str)
    qrels["relevance"] = pd.to_numeric(qrels["relevance"], errors="coerce").fillna(0).astype(int)
    return qrels

## 9. (Opsional) Weak Labels untuk Pilot Test
**Catatan:** label ini HANYA untuk uji coba awal, jangan dipakai sebagai satu-satunya ground truth untuk klaim publikasi/tesis.

In [12]:
def create_weak_qrels_lte(
    df: pd.DataFrame,
    query_id: str = "Q1",
) -> pd.DataFrame:
    """
    PILOT ONLY — DO NOT use as the sole ground truth for publication claims.

    More conservative than the original any(keyword) rule:
    requires Graph-RAG for Legal and Regulatory Reasoning,
    then assigns higher relevance when Norm-Aware and Temporal Regulatory
    are also present.
    """
    Core_terms = [
        "retrieval", "Augmented", "Generation", "RAG",
        "Graph-RAG", "GraphRAG", "Generative AI", "Large Language Models",
        "LLM", "HybridRAG", "Agentic RAG", "Natural Language Processing"
    ]
    structure_terms = [
        "Knowledge Graph", "ontology", "entities", "graph relation", "hierarchical graph",
        "Node", "Graph Neural Networks", "GNN", "temporal graphs"
    ]
    mechanisms_terms = [
        "vector search", "graph retrieval", "multi-hop retrieval",
        "embeddings", "neural network", "query expansion", "context compression"
    ]
    domain_terms = [
        "question answering", "QA", "legal", "cross-domain",
        "geospatial", "coverage map"
    ]

    rows = []
    for _, row in df.iterrows():
        text = (row["Title"] + " " + row["Abstract"]).lower()

        has_network = any(t in text for t in Core_terms)
        has_signal = any(t in text for t in structure_terms)
        has_ml = any(t in text for t in mechanisms_terms)
        has_spatial = any(t in text for t in domain_terms)

        relevance = 0
        if has_network and has_signal:
            relevance = 1
        if has_network and has_signal and has_ml:
            relevance = 2
        if has_network and has_signal and has_ml and has_spatial:
            relevance = 3

        rows.append({
            "query_id": query_id,
            "doc_id": row["doc_id"],
            "relevance": relevance,
        })

    return pd.DataFrame(rows)

## 10. Information Retrieval Metrics
Precision@K, Recall@K, MRR, nDCG@K.

In [13]:
def precision_at_k(
    ranked_doc_ids: List[str],
    relevance: Dict[str, int],
    k: int,
    threshold: int = 2,
) -> float:
    retrieved = ranked_doc_ids[:k]
    if not retrieved:
        return 0.0
    rel = sum(relevance.get(doc_id, 0) >= threshold for doc_id in retrieved)
    # Standard P@K denominator is K, but if fewer than K returned,
    # using len(retrieved) avoids artificial penalty for tiny corpora.
    return rel / len(retrieved)


def recall_at_k(
    ranked_doc_ids: List[str],
    relevance: Dict[str, int],
    k: int,
    threshold: int = 2,
) -> float:
    total_relevant = sum(v >= threshold for v in relevance.values())
    if total_relevant == 0:
        return 0.0
    retrieved_relevant = sum(
        relevance.get(doc_id, 0) >= threshold
        for doc_id in ranked_doc_ids[:k]
    )
    return retrieved_relevant / total_relevant


def reciprocal_rank(
    ranked_doc_ids: List[str],
    relevance: Dict[str, int],
    threshold: int = 2,
) -> float:
    for rank, doc_id in enumerate(ranked_doc_ids, start=1):
        if relevance.get(doc_id, 0) >= threshold:
            return 1.0 / rank
    return 0.0


def dcg_at_k(
    ranked_doc_ids: List[str],
    relevance: Dict[str, int],
    k: int,
) -> float:
    gains = [relevance.get(doc_id, 0) for doc_id in ranked_doc_ids[:k]]
    return sum(
        (2 ** rel - 1) / math.log2(rank + 1)
        for rank, rel in enumerate(gains, start=1)
    )


def ndcg_at_k(
    ranked_doc_ids: List[str],
    relevance: Dict[str, int],
    k: int,
) -> float:
    dcg = dcg_at_k(ranked_doc_ids, relevance, k)

    ideal_rels = sorted(relevance.values(), reverse=True)[:k]
    idcg = sum(
        (2 ** rel - 1) / math.log2(rank + 1)
        for rank, rel in enumerate(ideal_rels, start=1)
    )

    return dcg / idcg if idcg > 0 else 0.0


def evaluate_run(
    ranked_df: pd.DataFrame,
    query_id: str,
    qrels: pd.DataFrame,
    ks: Iterable[int] = (5, 10, 20),
    threshold: int = 2,
) -> Dict[str, float]:
    q = qrels[qrels["query_id"] == str(query_id)]
    relevance = dict(zip(q["doc_id"], q["relevance"]))
    ranked_ids = ranked_df["doc_id"].tolist()

    metrics: Dict[str, float] = {
        "MRR": reciprocal_rank(ranked_ids, relevance, threshold),
    }

    for k in ks:
        metrics[f"P@{k}"] = precision_at_k(ranked_ids, relevance, k, threshold)
        metrics[f"R@{k}"] = recall_at_k(ranked_ids, relevance, k, threshold)
        metrics[f"nDCG@{k}"] = ndcg_at_k(ranked_ids, relevance, k)

    return metrics

## 11. Query Set
Daftar query eksperimen. Tambahkan query lain (Q2, Q3, dst.) sesuai kebutuhan — pastikan setiap query memiliki qrels.

In [19]:
# Use multiple queries for publication-quality experiments.
# Each query must have relevance judgments in qrels.
QUERIES = pd.DataFrame([
    {
        "query_id": "Q1",
        "query": (
            "Graph-RAG for Legal and Regulatory Reasoning "
            "Norm-Aware and Temporal Regulatory"
        ),
    },
    # Example additional queries:
    # {
    #     "query_id": "Q2",
    #     "query": "machine learning cellular coverage prediction using RSRP and geospatial data",
    # },
    # {
    #     "query_id": "Q3",
    #     "query": "radio environment map construction from LTE drive test measurements",
    # },
])

## 12. Fungsi Eksperimen Lengkap (`run_experiment`)
Menjalankan seluruh pipeline: load corpus → BM25 → Dense → Hybrid RRF → Reranker → evaluasi metrik → simpan hasil.

In [20]:
def run_experiment(cfg: Config = CFG):
    # ---- Load corpus
    df = load_dimensions_csv(
        cfg.csv_path,
        skiprows=cfg.skiprows,
        encoding=cfg.encoding,
    )

    # Save document ID mapping so annotators can create qrels
    mapping_path = Path(cfg.output_dir) / "document_mapping_for_annotation.csv"
    df[
        [
            "doc_id", "Title", "Abstract", "PubYear",
            "Source title/Anthology title", "DOI", "Dimensions URL"
        ]
    ].to_csv(mapping_path, index=False)
    print(f"Annotation mapping saved: {mapping_path}")

    # ---- Build retrieval models once
    bm25 = BM25Retriever(df)

    dense = DenseFAISSRetriever(
        df=df,
        model_name=cfg.dense_model,
        batch_size=cfg.embedding_batch_size,
    )

    reranker = CrossEncoderReranker(
        model_name=cfg.reranker_model,
        batch_size=cfg.rerank_batch_size,
    )

    # ---- Load qrels if available, otherwise make WEAK pilot labels
    if os.path.exists(cfg.qrels_path):
        print(f"Loading manual qrels: {cfg.qrels_path}")
        qrels = load_qrels(cfg.qrels_path)
        qrels_source = "manual_qrels"
    else:
        warnings.warn(
            "Manual qrels not found. Using weak rule-based pilot labels. "
            "Do NOT use these alone for thesis/publication claims."
        )
        qrels_frames = []
        for qid in QUERIES["query_id"]:
            # Current weak-label function is tailored to LTE retrieval.
            qrels_frames.append(create_weak_qrels_lte(df, query_id=str(qid)))
        qrels = pd.concat(qrels_frames, ignore_index=True)
        qrels_source = "weak_pilot_qrels"

        weak_path = Path(cfg.output_dir) / "weak_qrels_DO_NOT_USE_AS_FINAL.csv"
        qrels.to_csv(weak_path, index=False)
        print(f"Weak qrels saved: {weak_path}")

    print("Qrels source:", qrels_source)

    # ---- Execute each query / retrieval method
    all_metrics = []
    all_results = []

    for _, qrow in QUERIES.iterrows():
        qid = str(qrow["query_id"])
        query = qrow["query"]

        print("\n" + "=" * 90)
        print(f"{qid}: {query}")
        print("=" * 90)

        bm25_res = bm25.search(query, cfg.bm25_top_k)
        dense_res = dense.search(query, cfg.dense_top_k)

        hybrid_res = reciprocal_rank_fusion(
            [bm25_res, dense_res],
            rrf_k=cfg.rrf_k,
            top_k=cfg.hybrid_top_k,
        )

        rerank_res = reranker.rerank(
            query=query,
            candidates=hybrid_res,
            corpus_df=df,
            top_k=cfg.rerank_top_k,
        )

        runs = {
            "BM25": bm25_res,
            "Dense_FAISS": dense_res,
            "Hybrid_RRF": hybrid_res,
            "Hybrid_RRF_Reranker": rerank_res,
        }

        # Save full ranked outputs
        for method, result in runs.items():
            tmp = result.copy()
            tmp["query_id"] = qid
            tmp["query"] = query
            tmp["method"] = method
            all_results.append(tmp)

            metrics = evaluate_run(
                ranked_df=result,
                query_id=qid,
                qrels=qrels,
                ks=cfg.eval_ks,
                threshold=cfg.binary_relevance_threshold,
            )

            metrics_row = {
                "query_id": qid,
                "method": method,
                "qrels_source": qrels_source,
                **metrics,
            }
            all_metrics.append(metrics_row)

    metrics_df = pd.DataFrame(all_metrics)
    results_df = pd.concat(all_results, ignore_index=True)

    # ---- Aggregate metrics across queries
    metric_cols = [
        c for c in metrics_df.columns
        if c.startswith(("P@", "R@", "nDCG@")) or c == "MRR"
    ]

    aggregate_df = (
        metrics_df
        .groupby("method", as_index=False)[metric_cols]
        .mean()
    )

    # ---- Add bibliographic metadata to retrieval output
    metadata_cols = [
        "doc_id", "Authors", "PubYear",
        "Source title/Anthology title", "DOI", "Dimensions URL"
    ]
    results_df = results_df.merge(
        df[metadata_cols],
        on="doc_id",
        how="left",
    )

    # Add relevance labels for each query result
    qrels_lookup = qrels[["query_id", "doc_id", "relevance"]].copy()
    results_df = results_df.merge(
        qrels_lookup,
        on=["query_id", "doc_id"],
        how="left",
    )
    results_df["relevance"] = results_df["relevance"].fillna(0).astype(int)

    # ---- Save outputs
    metrics_path = Path(cfg.output_dir) / "metrics_per_query.csv"
    aggregate_path = Path(cfg.output_dir) / "metrics_aggregate.csv"
    results_path = Path(cfg.output_dir) / "ranked_results_all_methods.csv"
    config_path = Path(cfg.output_dir) / "experiment_config.json"

    metrics_df.to_csv(metrics_path, index=False)
    aggregate_df.to_csv(aggregate_path, index=False)
    results_df.to_csv(results_path, index=False)

    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2, default=list)

    print("\nPER-QUERY METRICS")
    print(metrics_df.to_string(index=False))

    print("\nAGGREGATE METRICS")
    print(aggregate_df.to_string(index=False))

    print("\nSaved:")
    print(" -", metrics_path)
    print(" -", aggregate_path)
    print(" -", results_path)
    print(" -", config_path)

    return {
        "corpus": df,
        "qrels": qrels,
        "metrics_per_query": metrics_df,
        "metrics_aggregate": aggregate_df,
        "ranked_results": results_df,
    }

## 13. (Opsional) Uji Signifikansi untuk Eksperimen Multi-Query

In [21]:
def paired_randomization_test(
    x: np.ndarray,
    y: np.ndarray,
    n_iter: int = 10000,
    seed: int = 42,
) -> float:
    """
    Approximate paired randomization test.
    Useful only when you have MULTIPLE independent queries.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) != len(y):
        raise ValueError("x and y must have the same length")
    if len(x) < 2:
        raise ValueError("Need multiple queries for significance testing")

    rng = np.random.default_rng(seed)
    observed = abs(np.mean(x - y))
    diffs = x - y

    count = 0
    for _ in range(n_iter):
        signs = rng.choice([-1, 1], size=len(diffs))
        simulated = abs(np.mean(diffs * signs))
        if simulated >= observed:
            count += 1

    return (count + 1) / (n_iter + 1)

## 14. Jalankan Eksperimen
Pastikan `CFG.csv_path` dan `CFG.qrels_path` sudah menunjuk ke lokasi file yang benar di Google Drive Anda
(lihat bagian **Config** di atas) sebelum menjalankan sel ini.

In [22]:
results = run_experiment(CFG)

# Contoh akses hasil:
# results["metrics_aggregate"]
# results["ranked_results"].head(20)

Rows loaded       : 1682
Unique documents  : 1682
Duplicates removed: 0
Missing abstracts : 36
Missing DOI       : 29
Annotation mapping saved: /content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/Penulisan/retrieval_v2_results/document_mapping_for_annotation.csv


BM25 tokenize:   0%|          | 0/1682 [00:00<?, ?it/s]

Loading dense model: intfloat/e5-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding corpus...


Batches:   0%|          | 0/53 [00:00<?, ?it/s]

FAISS vectors: 1682, dimension: 768
Loading reranker: cross-encoder/ms-marco-MiniLM-L6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

/tmp/ipykernel_2869/631574348.py:39: UserWarning: Manual qrels not found. Using weak rule-based pilot labels. Do NOT use these alone for thesis/publication claims.
  warnings.warn(


Weak qrels saved: /content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/Penulisan/retrieval_v2_results/weak_qrels_DO_NOT_USE_AS_FINAL.csv
Qrels source: weak_pilot_qrels

Q1: Graph-RAG for Legal and Regulatory Reasoning Norm-Aware and Temporal Regulatory

PER-QUERY METRICS
query_id              method     qrels_source      MRR  P@5      R@5   nDCG@5  P@10     R@10  nDCG@10  P@20     R@20  nDCG@20
      Q1                BM25 weak_pilot_qrels 0.013889  0.0 0.000000 0.024226   0.0 0.000000 0.015721   0.0 0.000000 0.020824
      Q1         Dense_FAISS weak_pilot_qrels 0.142857  0.0 0.000000 0.079021   0.1 0.012987 0.092640   0.1 0.025974 0.093352
      Q1          Hybrid_RRF weak_pilot_qrels 0.111111  0.0 0.000000 0.073539   0.2 0.025974 0.103382   0.1 0.025974 0.073406
      Q1 Hybrid_RRF_Reranker weak_pilot_qrels 0.250000  0.2 0.012987 0.160365   0.2 0.025974 0.131332   0.2 0.025974 0.086353

AGGREGATE METRICS
             method      MRR  P@5      R@5   nDCG@5  P@10     R@10  nDCG

In [23]:
results["metrics_aggregate"]

,method,MRR,P@5,R@5,nDCG@5,P@10,R@10,nDCG@10,P@20,R@20,nDCG@20
0,BM25,0.013889,0.0,0.000000,0.024226,0.0,0.000000,0.015721,0.0,0.000000,0.020824
1,Dense_FAISS,0.142857,0.0,0.000000,0.079021,0.1,0.012987,0.092640,0.1,0.025974,0.093352
2,Hybrid_RRF,0.111111,0.0,0.000000,0.073539,0.2,0.025974,0.103382,0.1,0.025974,0.073406
3,Hybrid_RRF_Reranker,0.250000,0.2,0.012987,0.160365,0.2,0.025974,0.131332,0.2,0.025974,0.086353


In [24]:
results["ranked_results"].head(20)

,doc_id,Title,score,rank,method,query_id,query,Authors,PubYear,Source title/Anthology title,DOI,Dimensions URL,relevance
0,doi:10.48550/arxiv.2505.11946,Let's have a chat with the EU AI Act,25.657041,1,BM25,Q1,Graph-RAG for Legal and Regulatory Reasoning N...,"Kovari, Adam; Ghafourian, Yasin; Hegedus, Csab...",2025,arXiv,10.48550/arxiv.2505.11946,https://app.dimensions.ai/details/publication/...,0
1,doi:10.48550/arxiv.2605.29742,Citation-Closure Retrieval and Per-Rule Attrib...,24.335019,2,BM25,Q1,Graph-RAG for Legal and Regulatory Reasoning N...,"Ju, Yeong-Joon; Lee, Seong-Whan",2026,arXiv,10.48550/arxiv.2605.29742,https://app.dimensions.ai/details/publication/...,0
2,doi:10.48550/arxiv.2508.09893,RAGulating Compliance: A Multi-Agent Knowledge...,24.075517,3,BM25,Q1,Graph-RAG for Legal and Regulatory Reasoning N...,"Agarwal, Bhavik; Jomraj, Hemant Sunil; Kapluno...",2025,arXiv,10.48550/arxiv.2508.09893,https://app.dimensions.ai/details/publication/...,1
3,doi:10.48550/arxiv.2510.26309,GraphCompliance: Aligning Policy and Context G...,23.115277,4,BM25,Q1,Graph-RAG for Legal and Regulatory Reasoning N...,"Chung, Jiseong; Ko, Ronny; Yoo, Wonchul; Onizu...",2025,arXiv,10.48550/arxiv.2510.26309,https://app.dimensions.ai/details/publication/...,0
4,doi:10.3390/buildings16061224,Improving Access to Building Licensing Informa...,23.039822,5,BM25,Q1,Graph-RAG for Legal and Regulatory Reasoning N...,"Yan, Diya; Liu, Jiate; Han, Bocheng; Yang, Zhe...",2026,Buildings,10.3390/buildings16061224,https://app.dimensions.ai/details/publication/...,0
5,doi:10.25258/ijddt.16.41s.137,Neuro-Symbolic Legal Guardian: A Hybrid RAG an...,22.783276,6,BM25,Q1,Graph-RAG for Legal and Regulatory Reasoning N...,"B, Bharathi; V, Kavitha; A, Jeswin Prabhagaran...",2026,International Journal of Drug Delivery Technology,10.25258/ijddt.16.41s.137,https://app.dimensions.ai/details/publication/...,0
6,doi:10.20944/preprints202602.0640.v1,Improving Access to Building Licensing Informa...,22.263534,7,BM25,Q1,Graph-RAG for Legal and Regulatory Reasoning N...,"Yan, Diya; Liu, Jiate; Han, Bocheng; Yang, Zhe...",2026,Preprints.org,10.20944/preprints202602.0640.v1,https://app.dimensions.ai/details/publication/...,0
7,doi:10.48550/arxiv.2412.08593,Leveraging Graph-RAG and Prompt Engineering to...,22.054842,8,BM25,Q1,Graph-RAG for Legal and Regulatory Reasoning N...,"Masoudifard, Arsalan; Sorond, Mohammad Mowlavi...",2024,arXiv,10.48550/arxiv.2412.08593,https://app.dimensions.ai/details/publication/...,0
8,doi:10.48550/arxiv.2604.23585,ComplianceNLP: Knowledge-Graph-Augmented RAG f...,22.044913,9,BM25,Q1,Graph-RAG for Legal and Regulatory Reasoning N...,"Guo, Dongxin; Wu, Jikun; Yiu, Siu Ming",2026,arXiv,10.48550/arxiv.2604.23585,https://app.dimensions.ai/details/publication/...,0
9,doi:10.48550/arxiv.2603.13244,"Agentic AI, Retrieval-Augmented Generation, an...",21.499046,10,BM25,Q1,Graph-RAG for Legal and Regulatory Reasoning N...,"Osmond, Marcel",2026,arXiv,10.48550/arxiv.2603.13244,https://app.dimensions.ai/details/publication/...,0


## 15. Qrels Template Generator
Membuat CSV kosong untuk anotasi relevansi manual (0-3) oleh manusia.

In [25]:
def create_annotation_template(
    corpus_df: pd.DataFrame,
    query_id: str,
    candidate_doc_ids: Optional[List[str]] = None,
    output_path: str = "qrels_annotation_template.csv",
):
    """
    Creates a CSV for human relevance assessment.

    Recommended annotation:
      0 = Not relevant
      1 = Partially relevant
      2 = Relevant
      3 = Highly relevant

    For strong evaluation, use pooled candidates from BM25, Dense, Hybrid,
    and Reranked runs rather than judging only one method.
    """
    if candidate_doc_ids is None:
        sample = corpus_df.copy()
    else:
        sample = corpus_df[corpus_df["doc_id"].isin(candidate_doc_ids)].copy()

    out = sample[
        [
            "doc_id", "Title", "Abstract", "PubYear",
            "Source title/Anthology title", "DOI"
        ]
    ].copy()

    out.insert(0, "query_id", str(query_id))
    out["relevance"] = ""  # human annotator fills 0-3
    out["annotator_notes"] = ""

    out.to_csv(output_path, index=False)
    print(f"Annotation template saved: {output_path}")

## 16. Pooling untuk Penilaian Manual yang Adil (TREC-style)

In [26]:
def build_judgment_pool(
    run_frames: Dict[str, pd.DataFrame],
    pool_depth: int = 30,
) -> List[str]:
    """
    TREC-style pooling idea:
    combine top-N documents from every retrieval system,
    then manually judge the union. This reduces evaluation bias.
    """
    pooled = set()
    for _, frame in run_frames.items():
        pooled.update(frame.head(pool_depth)["doc_id"].tolist())
    return sorted(pooled)